In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder

import joblib
import os

os.makedirs('../models', exist_ok=True)

# Load dataset
df = pd.read_csv('../data/student_loans.csv')

# ---------------------------------------------------
# Label Encoding
# ---------------------------------------------------

le_gender = LabelEncoder()
le_course = LabelEncoder()
le_tier = LabelEncoder()
le_trend = LabelEncoder()

df['gender_enc'] = le_gender.fit_transform(df['gender'])

df['course_enc'] = le_course.fit_transform(df['course_type'])

df['tier_enc'] = le_tier.fit_transform(df['institute_tier'])

df['trend_enc'] = le_trend.fit_transform(df['gpa_trend'])

# ---------------------------------------------------
# 8 Engineered Features
# ---------------------------------------------------

df['family_support'] = (
    df['family_income_mo'] / (df['emi'] + 1)
)

df['loan_burden'] = (
    df['loan_amt'] / (df['family_income_mo'] * 12 + 1)
)

df['is_early_vintage'] = (
    df['months_since_disb'] <= 6
).astype(int)

df['payment_score'] = (
    (1 - df['missed_pmts_past'] / 10)
    *
    (1 - df['payment_delay_avg'] / 30)
)

df['academic_risk'] = (
    (df['gpa'] < 6.5)
    |
    (df['gpa_trend'] == 'Declining')
).astype(int)

df['employment_stress'] = (
    (df['employed'] == 0)
    &
    (df['emi_income'] > 0.45)
).astype(int)

df['support_score'] = (
    df['co_borrower']
    +
    df['auto_debit']
)

df['income_adequacy'] = (
    df['salary_mo'] / (df['emi'] + 1)
)

# ---------------------------------------------------
# Print New Features
# ---------------------------------------------------

print("8 NEW FEATURES CREATED:")

new_feats = [
    'family_support',
    'loan_burden',
    'is_early_vintage',
    'payment_score',
    'academic_risk',
    'employment_stress',
    'support_score',
    'income_adequacy'
]

for f in new_feats:
    print(f"{f}: mean = {df[f].mean():.4f}")

# ---------------------------------------------------
# Save Encoders
# ---------------------------------------------------

for name, enc in [
    ('le_gender', le_gender),
    ('le_course', le_course),
    ('le_tier', le_tier),
    ('le_trend', le_trend)
]:
    joblib.dump(enc, f'../models/{name}.pkl')

# ---------------------------------------------------
# Save Engineered Dataset
# ---------------------------------------------------

df.to_csv(
    '../data/student_loans_engineered.csv',
    index=False
)

print()
print("Saved: data/student_loans_engineered.csv")

8 NEW FEATURES CREATED:
family_support: mean = 17.1888
loan_burden: mean = 0.8295
is_early_vintage: mean = 0.1023
payment_score: mean = 0.8006
academic_risk: mean = 0.4155
employment_stress: mean = 0.1748
support_score: mean = 1.2330
income_adequacy: mean = 7.2054

Saved: data/student_loans_engineered.csv
